# Prompt 8: Spark SQL Fundamentals — Querying DataFrames
## Databricks Certified Associate Developer for Apache Spark
### Topic 2 — Using Spark SQL (20%)

---

Spark SQL lets you query DataFrames using familiar SQL syntax **without leaving PySpark**.
The SQL engine and the DataFrame API compile down to the **same physical execution plan** —
there is no performance difference between them for equivalent queries.

**Exam facts to memorise:**
- `createOrReplaceTempView()` → session-scoped view; only visible in the current SparkSession
- `createGlobalTempView()` → cross-session view; accessed via `global_temp.<view_name>`
- `spark.sql('SELECT ...')` always returns a **DataFrame**
- SQL and DataFrame API are **fully interchangeable** in the same pipeline
- Views are **logical** only — they store no data, just the query definition
- `createOrReplaceTempView` silently replaces an existing view; `createTempView` raises an error if the name already exists

## 1. Registering Views

### Local Temporary View vs Global Temporary View

| Feature | Local Temp View | Global Temp View |
|---------|----------------|------------------|
| Method | `df.createOrReplaceTempView('name')` | `df.createGlobalTempView('name')` |
| Scope | Current SparkSession only | All SparkSessions in the same application |
| Access in SQL | `SELECT * FROM name` | `SELECT * FROM global_temp.name` |
| Lifetime | Until SparkSession is closed | Until Spark application stops or view is dropped |
| Replace safely | `createOrReplaceTempView` | `createOrReplaceGlobalTempView` |
| Error on duplicate | `createTempView` raises `AnalysisException` | `createGlobalTempView` raises `AnalysisException` |

```python
# Local view — accessible as just 'employees' in this SparkSession
df.createOrReplaceTempView('employees')
spark.sql('SELECT * FROM employees').show()

# Global view — accessible as 'global_temp.employees' from any session in this app
df.createOrReplaceGlobalTempView('employees_global')
spark.sql('SELECT * FROM global_temp.employees_global').show()

# Drop views manually
spark.catalog.dropTempView('employees')
spark.catalog.dropGlobalTempView('employees_global')
```

### View Management via Catalog API

```python
spark.catalog.listTables()          # lists all views in the current session
spark.catalog.tableExists('name')   # check if a view/table exists
spark.catalog.dropTempView('name')  # drop local temp view
spark.catalog.dropGlobalTempView('name')  # drop global temp view
```

## 2. SQL ↔ DataFrame API Equivalents

| SQL Clause | DataFrame API | Notes |
|-----------|--------------|-------|
| `SELECT col1, col2` | `df.select('col1', 'col2')` | |
| `SELECT col AS alias` | `df.select(col('col').alias('alias'))` | |
| `WHERE condition` | `df.filter(condition)` or `df.where(condition)` | `filter` and `where` are aliases |
| `GROUP BY col` | `df.groupBy('col').agg(...)` | |
| `HAVING count > 5` | `.filter(col('count') > 5)` after `.agg()` | No direct `.having()` method |
| `ORDER BY col DESC` | `df.orderBy(col('col').desc())` | |
| `LIMIT n` | `df.limit(n)` | |
| `JOIN ... ON` | `df.join(other, on='col', how='inner')` | |
| `DISTINCT` | `df.distinct()` | |
| `UNION ALL` | `df.union(other)` | Note: `union()` does NOT deduplicate |
| `UNION` (distinct) | `df.union(other).distinct()` | |
| `CASE WHEN` | `when(cond, val).when(...).otherwise(...)` | |
| `COUNT(*)` | `df.count()` or `F.count('*')` | |
| `COUNT(DISTINCT col)` | `F.countDistinct('col')` | |

> **When to choose SQL vs DataFrame API:**
> - **SQL**: large teams familiar with SQL, complex multi-table queries, CASE WHEN logic, reporting/BI users
> - **DataFrame API**: dynamic query building in Python, chaining transformations programmatically, type safety, unit testing
> - Both produce **identical execution plans** — use whichever is more readable in context

In [ ]:
# Cell 1: Setup — create sample DataFrames and register views
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as F_sum, desc, when, lit

spark = SparkSession.builder \
    .appName('SparkSQLFundamentals') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Employees dataset
employees_data = [
    (1, 'Alice',   'Engineering', 95000, 'London',    2019),
    (2, 'Bob',     'Marketing',   72000, 'New York',  2018),
    (3, 'Carol',   'Engineering', 110000,'London',    2016),
    (4, 'Dave',    'HR',          65000, 'Chicago',   2021),
    (5, 'Eve',     'Engineering', 105000,'Berlin',    2017),
    (6, 'Frank',   'Marketing',   78000, 'New York',  2020),
    (7, 'Grace',   'HR',          68000, 'Chicago',   2019),
    (8, 'Henry',   'Engineering', 88000, 'London',    2022),
    (9, 'Iris',    'Marketing',   82000, 'Berlin',    2018),
    (10,'James',   'Engineering', 120000,'New York',  2015),
]
emp_cols = ['id', 'name', 'department', 'salary', 'city', 'hire_year']
emp_df = spark.createDataFrame(employees_data, emp_cols)

# Departments dataset
dept_data = [
    ('Engineering', 'Alice Smith',  5_000_000),
    ('Marketing',   'Bob Jones',    2_000_000),
    ('HR',          'Carol White',  1_000_000),
    ('Finance',     'Dave Brown',   3_000_000),
]
dept_df = spark.createDataFrame(dept_data, ['dept_name', 'manager', 'budget'])

# Register as local temporary views
emp_df.createOrReplaceTempView('employees')
dept_df.createOrReplaceTempView('departments')

print('Views registered:')
for t in spark.catalog.listTables():
    print(f'  {t.name} (tableType={t.tableType}, isTemporary={t.isTemporary})')

print('\nEmployees sample:')
emp_df.show(5)

In [ ]:
# Cell 2: Example 1 — SELECT with column aliases
# SQL
print('=== SQL: SELECT with aliases ===')
spark.sql("""
    SELECT
        name,
        department AS dept,
        salary,
        salary * 1.1 AS salary_with_raise
    FROM employees
    WHERE department = 'Engineering'
""").show()

# DataFrame API equivalent
print('=== DataFrame API equivalent ===')
emp_df.filter(col('department') == 'Engineering') \
    .select(
        col('name'),
        col('department').alias('dept'),
        col('salary'),
        (col('salary') * 1.1).alias('salary_with_raise')
    ).show()

In [ ]:
# Cell 3: Example 2 — WHERE with multiple conditions
print('=== SQL: WHERE with AND / OR ===')
spark.sql("""
    SELECT name, department, salary, city
    FROM employees
    WHERE (department = 'Engineering' OR department = 'Marketing')
      AND salary > 80000
    ORDER BY salary DESC
""").show()

# DataFrame API equivalent
print('=== DataFrame API equivalent ===')
emp_df.filter(
    (col('department').isin('Engineering', 'Marketing')) & (col('salary') > 80000)
).select('name', 'department', 'salary', 'city') \
 .orderBy(col('salary').desc()) \
 .show()

In [ ]:
# Cell 4: Example 3 — GROUP BY with HAVING
print('=== SQL: GROUP BY with HAVING ===')
spark.sql("""
    SELECT
        department,
        COUNT(*) AS headcount,
        ROUND(AVG(salary), 2) AS avg_salary,
        MAX(salary) AS max_salary,
        MIN(salary) AS min_salary
    FROM employees
    GROUP BY department
    HAVING COUNT(*) >= 2
    ORDER BY avg_salary DESC
""").show()

# DataFrame API equivalent — note: no direct HAVING; use filter() after agg()
from pyspark.sql.functions import round as F_round, max as F_max, min as F_min

print('=== DataFrame API equivalent ===')
emp_df.groupBy('department') \
    .agg(
        count('*').alias('headcount'),
        F_round(avg('salary'), 2).alias('avg_salary'),
        F_max('salary').alias('max_salary'),
        F_min('salary').alias('min_salary')
    ) \
    .filter(col('headcount') >= 2) \
    .orderBy(col('avg_salary').desc()) \
    .show()

In [ ]:
# Cell 5: Example 4 — ORDER BY with LIMIT + JOIN
print('=== SQL: JOIN employees with departments, ORDER BY, LIMIT ===')
spark.sql("""
    SELECT
        e.name,
        e.salary,
        d.dept_name,
        d.budget,
        ROUND(e.salary / d.budget * 100, 4) AS salary_pct_of_budget
    FROM employees e
    INNER JOIN departments d ON e.department = d.dept_name
    ORDER BY salary DESC
    LIMIT 5
""").show()

# DataFrame API equivalent
from pyspark.sql.functions import round as F_round

print('=== DataFrame API equivalent ===')
emp_df.alias('e') \
    .join(dept_df.alias('d'), col('e.department') == col('d.dept_name'), how='inner') \
    .select(
        col('e.name'),
        col('e.salary'),
        col('d.dept_name'),
        col('d.budget'),
        F_round(col('e.salary') / col('d.budget') * 100, 4).alias('salary_pct_of_budget')
    ) \
    .orderBy(col('salary').desc()) \
    .limit(5) \
    .show()

In [ ]:
# Cell 6: Example 5 — Subquery
print('=== SQL: Subquery — employees earning above their dept average ===')
spark.sql("""
    SELECT e.name, e.department, e.salary, dept_avg.avg_sal
    FROM employees e
    INNER JOIN (
        SELECT department, ROUND(AVG(salary), 0) AS avg_sal
        FROM employees
        GROUP BY department
    ) dept_avg
    ON e.department = dept_avg.department
    WHERE e.salary > dept_avg.avg_sal
    ORDER BY e.department, e.salary DESC
""").show()

# DataFrame API equivalent
print('=== DataFrame API equivalent ===')
dept_avg_df = emp_df.groupBy('department') \
    .agg(F_round(avg('salary'), 0).alias('avg_sal'))

emp_df.join(dept_avg_df, on='department') \
    .filter(col('salary') > col('avg_sal')) \
    .select('name', 'department', 'salary', 'avg_sal') \
    .orderBy('department', col('salary').desc()) \
    .show()

In [ ]:
# Cell 7: Example 6 — CASE WHEN (conditional expressions)
print('=== SQL: CASE WHEN — salary band classification ===')
spark.sql("""
    SELECT
        name,
        department,
        salary,
        CASE
            WHEN salary >= 100000 THEN 'Band A (Senior)'
            WHEN salary >= 80000  THEN 'Band B (Mid)'
            WHEN salary >= 65000  THEN 'Band C (Junior)'
            ELSE                       'Band D (Entry)'
        END AS salary_band,
        CASE
            WHEN hire_year <= 2017 THEN 'Veteran (7+ years)'
            WHEN hire_year <= 2020 THEN 'Experienced (3-6 years)'
            ELSE                       'New Hire (<3 years)'
        END AS tenure_group
    FROM employees
    ORDER BY salary DESC
""").show(truncate=False)

# DataFrame API equivalent — when().when().otherwise()
print('=== DataFrame API equivalent ===')
emp_df.withColumn('salary_band',
    when(col('salary') >= 100000, 'Band A (Senior)')
    .when(col('salary') >= 80000,  'Band B (Mid)')
    .when(col('salary') >= 65000,  'Band C (Junior)')
    .otherwise('Band D (Entry)')
).withColumn('tenure_group',
    when(col('hire_year') <= 2017, 'Veteran (7+ years)')
    .when(col('hire_year') <= 2020, 'Experienced (3-6 years)')
    .otherwise('New Hire (<3 years)')
).select('name', 'department', 'salary', 'salary_band', 'tenure_group') \
 .orderBy(col('salary').desc()) \
 .show(truncate=False)

In [ ]:
# Cell 8: Example 7 — Global Temporary View (cross-session access)
print('=== Global Temporary View demo ===')
emp_df.createOrReplaceGlobalTempView('employees_global')

# Access via global_temp.<name> — note the database prefix
result = spark.sql('SELECT department, COUNT(*) AS cnt FROM global_temp.employees_global GROUP BY department')
result.show()

# Simulate another SparkSession accessing the global view
spark2 = SparkSession.builder \
    .appName('AnotherSession') \
    .master('local[*]') \
    .getOrCreate()

# spark2 can see global_temp views from the same Spark application
print('Accessed from spark2 (different session, same app):')
spark2.sql('SELECT COUNT(*) AS total FROM global_temp.employees_global').show()

# Local temp views are NOT shared across sessions
print('\nLocal temp views in spark2 (should be empty):')
print([t.name for t in spark2.catalog.listTables() if t.isTemporary and t.database != 'global_temp'])

print()
print('Key exam fact:')
print('  createOrReplaceTempView  → session-scoped → access as: SELECT * FROM <name>')
print('  createGlobalTempView     → app-scoped     → access as: SELECT * FROM global_temp.<name>')

In [ ]:
# Cell 9: Example 8 — Mixing SQL and DataFrame API in one pipeline
print('=== Mixed SQL + DataFrame API pipeline ===')

# Step 1: Use SQL to get department averages
dept_stats = spark.sql("""
    SELECT
        department,
        COUNT(*)               AS headcount,
        ROUND(AVG(salary), 0)  AS avg_salary,
        SUM(salary)            AS total_salary
    FROM employees
    GROUP BY department
""")

# Step 2: Register the SQL result as another view, continue with DataFrame API
dept_stats.createOrReplaceTempView('dept_stats')

# Step 3: Join back using DataFrame API
final = emp_df.join(dept_stats, on='department') \
    .withColumn('above_avg', when(col('salary') > col('avg_salary'), 'Yes').otherwise('No')) \
    .select('name', 'department', 'salary', 'avg_salary', 'above_avg') \
    .orderBy('department', desc('salary'))

final.show()

# Step 4: Continue back in SQL using the final view
final.createOrReplaceTempView('enriched_employees')
spark.sql("""
    SELECT department,
           SUM(CASE WHEN above_avg = 'Yes' THEN 1 ELSE 0 END) AS above_avg_count,
           COUNT(*) AS total
    FROM enriched_employees
    GROUP BY department
""").show()

spark.stop()
print('SparkSession stopped.')

## 3. Spark SQL Patterns — Quick Reference

### Registering and Using Views

```python
# Pattern 1: Register → SQL → get DataFrame back
df.createOrReplaceTempView('my_view')
result_df = spark.sql('SELECT col1, SUM(col2) AS total FROM my_view GROUP BY col1')
result_df.show()  # result_df is a standard DataFrame

# Pattern 2: SQL result → register → more SQL
spark.sql('SELECT ... FROM my_view').createOrReplaceTempView('step2_view')
spark.sql('SELECT ... FROM step2_view WHERE ...').show()

# Pattern 3: Using selectExpr() for inline SQL expressions without full spark.sql()
df.selectExpr(
    'name',
    'salary * 1.1 AS salary_with_raise',
    'CASE WHEN salary > 100000 THEN "Senior" ELSE "Junior" END AS level'
).show()
```

### Multi-line SQL String Best Practices

```python
# Use triple-quoted strings for readable SQL
query = """
    SELECT
        department,
        COUNT(*)   AS headcount,
        AVG(salary) AS avg_salary
    FROM employees
    WHERE hire_year >= 2018
    GROUP BY department
    HAVING COUNT(*) >= 2
    ORDER BY avg_salary DESC
"""
spark.sql(query).show()

# Dynamic SQL with Python f-strings (be careful of SQL injection in production)
dept = 'Engineering'
spark.sql(f"SELECT * FROM employees WHERE department = '{dept}'").show()
# Safer: use DataFrame API for user-supplied values
emp_df.filter(col('department') == dept).show()
```

## 4. Real-World Scenarios

<details>
<summary>Scenario 1: Analyst Team Using Spark SQL Views for Ad-Hoc Reporting</summary>

**Situation:**
A data analytics team has analysts who are SQL experts but not PySpark developers.
The data engineering team prepares cleaned DataFrames, registers them as temp views,
and lets analysts write standard SQL against them in Databricks notebooks.

**Engineer sets up views:**
```python
# Data engineer registers cleaned data as views
orders_clean_df.createOrReplaceTempView('orders')
customers_df.createOrReplaceTempView('customers')
products_df.createOrReplaceTempView('products')

print('Views available:', [t.name for t in spark.catalog.listTables()])
# ['orders', 'customers', 'products']
```

**Analyst writes SQL queries (no PySpark knowledge needed):**
```sql
-- Monthly revenue by product category
SELECT
    p.category,
    DATE_FORMAT(o.order_date, 'yyyy-MM') AS month,
    SUM(o.quantity * o.unit_price) AS revenue,
    COUNT(DISTINCT o.customer_id) AS unique_customers
FROM orders o
JOIN products p ON o.product_id = p.product_id
WHERE o.order_date >= '2024-01-01'
GROUP BY p.category, DATE_FORMAT(o.order_date, 'yyyy-MM')
HAVING SUM(o.quantity * o.unit_price) > 10000
ORDER BY month, revenue DESC
```

**Expected Outcome:**
Analysts get self-service access to production-quality data without needing to learn
the DataFrame API. The engineer controls data quality at the view registration point.

**Exam Sub-topic:** `createOrReplaceTempView()`; `spark.sql()`; SQL vs DataFrame API
</details>

<details>
<summary>Scenario 2: Global Temp Views for Cross-Notebook Shared Reference Data</summary>

**Situation:**
A Databricks workspace has a shared lookup table (country codes, currency rates)
that multiple notebooks in the same Spark application need to access.
Using a global temporary view allows one notebook to populate it and others to query it.

**Notebook 1 (data loader):**
```python
currency_rates = spark.read.parquet('/shared/currency_rates/')

# Register as GLOBAL view — accessible from all notebooks in this cluster session
currency_rates.createOrReplaceGlobalTempView('currency_rates')
print('Global view registered. Other notebooks can access global_temp.currency_rates')
```

**Notebook 2 (uses the shared view — different SparkSession):**
```python
# Must prefix with global_temp. — this is a common exam gotcha!
converted = spark.sql("""
    SELECT
        t.transaction_id,
        t.amount_usd,
        t.currency,
        t.amount_usd * cr.rate AS amount_local
    FROM transactions t
    JOIN global_temp.currency_rates cr ON t.currency = cr.currency_code
""")
converted.show()
```

**Expected Outcome:**
The currency rates are loaded once and shared across the entire Spark application's sessions.
No re-reading from storage in each notebook.

**Exam Sub-topic:** `createGlobalTempView()`; `global_temp` database prefix; view lifetime
</details>

<details>
<summary>Scenario 3: Dynamic SQL Query Building Using DataFrame API</summary>

**Situation:**
A data platform team builds a parameterised reporting tool where users can choose
which departments to include, a salary threshold, and which columns to display.
This is best done with the DataFrame API (dynamic column list, filter construction)
rather than SQL string concatenation (SQL injection risk).

**Code:**
```python
def get_employee_report(spark, departments, min_salary, output_cols):
    """
    Dynamic report — safer than f-string SQL with user inputs.
    departments: list of dept names, e.g. ['Engineering', 'Marketing']
    min_salary: int threshold
    output_cols: list of column names to include
    """
    return (
        emp_df
        .filter(col('department').isin(departments))       # dynamic filter list
        .filter(col('salary') >= min_salary)               # parameterised threshold
        .select([col(c) for c in output_cols])             # dynamic column selection
        .orderBy(col('salary').desc())
    )

# User-specified parameters (from UI/API request)
result = get_employee_report(
    spark,
    departments=['Engineering', 'HR'],
    min_salary=70000,
    output_cols=['name', 'department', 'salary', 'city']
)
result.show()
```

**Expected Outcome:**
Safe, reusable parameterised report that cannot be SQL-injected. Output changes based
on runtime parameters without modifying code.

**Exam Sub-topic:** DataFrame API vs SQL for dynamic queries; `isin()`; dynamic column selection
</details>

<details>
<summary>Scenario 4: Mixing SQL and DataFrame API in an ETL Pipeline</summary>

**Situation:**
An ETL pipeline has a complex business rule (written by a SQL developer) for fraud scoring,
but the result needs further DataFrame API transformation before writing to the output.

**Code:**
```python
# Step 1: Load raw data with DataFrame API
transactions = spark.read.parquet('/data/transactions/')
transactions.createOrReplaceTempView('transactions')

# Step 2: Apply complex scoring rules in SQL (business analyst wrote this)
scored = spark.sql("""
    SELECT
        transaction_id,
        customer_id,
        amount,
        CASE
            WHEN amount > 10000 AND country != home_country THEN 90
            WHEN amount > 5000  AND hour(ts) BETWEEN 0 AND 5 THEN 70
            WHEN amount > 1000  AND days_since_last_txn > 30  THEN 50
            ELSE 10
        END AS fraud_score
    FROM transactions
""")

# Step 3: Continue with DataFrame API for ML feature engineering
features = scored \
    .withColumn('is_high_risk', col('fraud_score') >= 70) \
    .withColumn('score_bucket', (col('fraud_score') / 10).cast('int') * 10)

# Step 4: Write using DataFrame API
features.write.partitionBy('score_bucket').parquet('/output/fraud_features/')
```

**Expected Outcome:**
SQL and DataFrame API work in the same pipeline. The SQL result is a regular DataFrame
that can be further transformed. The final write partitions output by score bucket.

**Exam Sub-topic:** Mixing SQL + DataFrame API; `createOrReplaceTempView`; `spark.sql()` returns DataFrame
</details>

<details>
<summary>Scenario 5: Using selectExpr() for Inline SQL Without Full Temp View Registration</summary>

**Situation:**
A developer wants to apply SQL expressions to a DataFrame without the overhead of
registering a temp view. `selectExpr()` allows inline SQL-style expressions
applied directly to any DataFrame.

**Code:**
```python
# selectExpr() — each argument is a SQL expression string
result = emp_df.selectExpr(
    'name',
    'department',
    'salary',
    'salary * 1.15 AS salary_with_bonus',
    'UPPER(city) AS city_upper',
    'CASE WHEN hire_year < 2019 THEN true ELSE false END AS is_veteran',
    'YEAR(CURRENT_DATE()) - hire_year AS years_of_service'
)
result.show(truncate=False)
result.printSchema()

# selectExpr equivalences:
#   selectExpr('col * 2 AS double_col')   ≡   select(expr('col * 2').alias('double_col'))
#   selectExpr('UPPER(name)')             ≡   select(upper(col('name')))
#   selectExpr('*')                       ≡   select('*') → all columns
#   df.selectExpr('* EXCEPT (col)')       ≡   df.drop('col')
```

**Expected Outcome:**
`selectExpr()` is a concise alternative to registering a view then querying it.
It accepts any valid SQL expression and returns a standard DataFrame.
Useful for quick column transformations using SQL syntax without importing functions.

**Exam Sub-topic:** `selectExpr()`; SQL expressions on DataFrames; `expr()` function
</details>

## 5. Exam Cheat Sheet

| Concept | Key Point |
|---------|----------|
| `createOrReplaceTempView()` | Session-scoped; replaces silently; access as `SELECT * FROM name` |
| `createTempView()` | Session-scoped; raises `AnalysisException` if name already exists |
| `createGlobalTempView()` | App-scoped; access as `SELECT * FROM global_temp.name` |
| `createOrReplaceGlobalTempView()` | App-scoped; replaces silently |
| `spark.sql(query)` | Always returns a **DataFrame** |
| Views store data? | **No** — views are logical only; they re-execute the query each time |
| SQL vs DataFrame API performance | **Identical** — same Catalyst optimizer; same physical plan |
| `selectExpr()` | Apply SQL expressions to a DataFrame without registering a view |
| HAVING equivalent in DataFrame API | `.filter()` **after** `.groupBy().agg()` |
| `filter()` vs `where()` | **Aliases** — identical behaviour |
| `union()` in DataFrame API | Equivalent to SQL `UNION ALL` (does NOT deduplicate) |
| SQL UNION (distinct) | `df1.union(df2).distinct()` |
| `global_temp` database | Auto-created by Spark; holds all global temp views |

---

## 6. Practice Q&A

<details>
<summary>Q1: What is the difference between createOrReplaceTempView and createGlobalTempView?</summary>

**A:**
- `createOrReplaceTempView('name')`: Registers a **session-scoped** view. Only visible within the current SparkSession. Queried as `SELECT * FROM name`. Automatically dropped when the SparkSession closes.
- `createGlobalTempView('name')`: Registers an **application-scoped** view. Visible across all SparkSessions in the same Spark application. Queried as `SELECT * FROM global_temp.name` (must use the `global_temp` database prefix). Dropped when the Spark application terminates.

Key exam gotcha: forgetting the `global_temp.` prefix when querying a global view will result in `AnalysisException: Table or view not found`.
</details>

<details>
<summary>Q2: What does spark.sql() always return?</summary>

**A:** `spark.sql()` always returns a **DataFrame**. This means you can chain DataFrame API operations directly on the result:
```python
spark.sql('SELECT * FROM employees').filter(col('salary') > 80000).show()
spark.sql('SELECT * FROM employees').createOrReplaceTempView('high_earners_base')
```
The DataFrame is **lazily evaluated** — no actual computation happens until an action is called.
</details>

<details>
<summary>Q3: Is there a performance difference between SQL and the DataFrame API?</summary>

**A:** **No.** Both SQL and the DataFrame API go through the same **Catalyst optimizer pipeline**:
1. Parse → Analyze → Logical Plan Optimization → Physical Planning → Code Generation

An equivalent SQL query and DataFrame API query produce **exactly the same execution plan**. The choice is purely about readability, team familiarity, and programmatic flexibility (dynamic query building is easier with the DataFrame API).
</details>

<details>
<summary>Q4: How do you implement HAVING in the DataFrame API?</summary>

**A:** The DataFrame API has no `.having()` method. Instead, apply `.filter()` **after** `.groupBy().agg()`:
```python
# SQL HAVING equivalent
df.groupBy('department') \
  .agg(count('*').alias('headcount'), avg('salary').alias('avg_sal')) \
  .filter(col('headcount') >= 3)   # ← this is HAVING
  .show()

# Equivalent SQL:
# SELECT department, COUNT(*) AS headcount, AVG(salary) AS avg_sal
# FROM employees GROUP BY department HAVING COUNT(*) >= 3
```
</details>

<details>
<summary>Q5: When should you choose the DataFrame API over SQL?</summary>

**A:** Prefer the **DataFrame API** when:
1. **Dynamic queries** — column lists, filter conditions, or table names are determined at runtime (avoid f-string SQL injection risk)
2. **Chaining complex transformations** programmatically — easier to read and test in Python
3. **Unit testing** — DataFrame operations are easier to mock and test than SQL strings
4. **Team uses Python/Scala** as primary language

Prefer **SQL** when:
1. Team has strong SQL background (analysts, data scientists)
2. Complex multi-table joins, CTEs, window functions are easier to express in SQL
3. Integrating with BI tools or notebooks where SQL is the native interface
4. The query is relatively static and readability in SQL is clearer
</details>